# Experiment 5: TREC 2021 External Gold Validation

**Purpose:** This experiment directly addresses whether models trained on
Jaccard-based silver labels (Experiment 3) produce rankings that align with
independent expert-adjudicated gold labels from TREC 2021.

**Design:** A BioBERT cross-encoder trained on eICU/Jaccard silver labels
is evaluated against TREC 2021 Clinical Trials gold judgments. Rankings under
silver and gold evaluation are compared to compute an empirical SRI.

In [ ]:
# =========================
# CONFIGURATION
# =========================
import os, sys
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)

_nb_dir = Path('.').resolve()
_repo_root = _nb_dir.parent if _nb_dir.name == 'notebooks' else _nb_dir

try:
    sys.path.insert(0, str(_repo_root / 'config'))
    from config_paths_private import *
    CONFIG_MODE = 'private'
    print('[INFO] Using private path configuration')
except ImportError:
    REPO_ROOT = _repo_root
    RUN_ID = 'public_run'
    DATADIR = REPO_ROOT / 'data'
    OUTDIR = REPO_ROOT / 'outputs'
    CONFIG_MODE = 'public'
    print('[INFO] Using public path configuration')

Path(OUTDIR).mkdir(parents=True, exist_ok=True)


In [ ]:
# =========================
# SHARED UTILITIES
# =========================
import os, json, time, platform, hashlib, random
from pathlib import Path
import numpy as np
import pandas as pd

def set_global_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)

def now_utc_compact():
    import datetime
    return datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%d_%H%M%S_utc')

def safe_mkdir(p):
    Path(p).mkdir(parents=True, exist_ok=True)
    return Path(p)

def make_run_dirs(exp_name):
    run_id = f'{exp_name}_{now_utc_compact()}'
    base = Path('outputs') / exp_name / run_id
    figs = base / 'figures'
    tabs = base / 'tables'
    logs = base / 'logs'
    for d in [base, figs, tabs, logs]:
        safe_mkdir(d)
    return run_id, base, figs, tabs, logs

def write_manifest(base, meta):
    mf = Path(base) / 'run_manifest.json'
    meta = dict(meta)
    meta['platform'] = {'python': platform.python_version(), 'platform': platform.platform()}
    mf.write_text(json.dumps(meta, indent=2))
    return mf

set_global_seed(42)


In [ ]:
# =========================
# 05 - LOAD TREC 2021 GOLD DATA
# =========================
EXP = '05_trec_validation'
run_id, out_base, out_figs, out_tabs, out_logs = make_run_dirs(EXP)

from sklearn.metrics import ndcg_score, roc_auc_score, average_precision_score

TREC_CHECKPOINTS = Path('/content/drive/MyDrive/vv_health/trec/trec2021_orthogonal_validation_v3/checkpoints')

USE_TREC = False
df_preds = None

try:
    checkpoint_preds = TREC_CHECKPOINTS / 'checkpoint_05_predictions.parquet'
    if checkpoint_preds.exists():
        df_preds = pd.read_parquet(checkpoint_preds)
        USE_TREC = True
        print(f'[INFO] Loaded TREC predictions from checkpoint')
        print(f'       Pairs: {len(df_preds):,}')
        print(f'       Patients: {df_preds["patient_id"].nunique()}')
        print(f'       Columns: {list(df_preds.columns)}')
        print(f'       Gold positive rate: {df_preds["label"].mean():.3f}')
    else:
        print('[INFO] TREC predictions not found. Skipping external validation.')
except Exception as e:
    print(f'[WARN] Could not load TREC data: {e}')
    USE_TREC = False


In [ ]:
# =========================
# 05 - COMPUTE RANKINGS UNDER GOLD EVALUATION
# =========================
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

if USE_TREC and df_preds is not None and 'biobert_score' in df_preds.columns:
    gold_labels = df_preds['label'].values
    biobert_scores = df_preds['biobert_score'].values
    bm25_scores = df_preds['bm25_score'].values

    # Per-patient ranking metrics under GOLD labels
    patients = df_preds['patient_id'].unique()
    biobert_ndcg_gold = []
    bm25_ndcg_gold = []
    biobert_map_gold = []
    bm25_map_gold = []

    for pid in patients:
        mask = df_preds['patient_id'] == pid
        y_true = gold_labels[mask]
        if y_true.sum() == 0 or y_true.sum() == len(y_true):
            continue
        bb_s = biobert_scores[mask]
        bm_s = bm25_scores[mask]
        try:
            biobert_ndcg_gold.append(ndcg_score([y_true], [bb_s], k=10))
            bm25_ndcg_gold.append(ndcg_score([y_true], [bm_s], k=10))
        except Exception:
            pass
        try:
            biobert_map_gold.append(average_precision_score(y_true, bb_s))
            bm25_map_gold.append(average_precision_score(y_true, bm_s))
        except Exception:
            pass

    biobert_ndcg_mean = np.mean(biobert_ndcg_gold) if biobert_ndcg_gold else 0
    bm25_ndcg_mean = np.mean(bm25_ndcg_gold) if bm25_ndcg_gold else 0
    biobert_map_mean = np.mean(biobert_map_gold) if biobert_map_gold else 0
    bm25_map_mean = np.mean(bm25_map_gold) if bm25_map_gold else 0

    try:
        biobert_auc = roc_auc_score(gold_labels, biobert_scores)
        bm25_auc = roc_auc_score(gold_labels, bm25_scores)
    except Exception:
        biobert_auc = 0
        bm25_auc = 0

    print('=== TREC 2021 Gold-Label Evaluation ===')
    print(f'  BioBERT AUC:      {biobert_auc:.4f}')
    print(f'  BM25 AUC:         {bm25_auc:.4f}')
    print(f'  BioBERT NDCG@10:  {biobert_ndcg_mean:.4f}')
    print(f'  BM25 NDCG@10:     {bm25_ndcg_mean:.4f}')
    print(f'  BioBERT MAP:      {biobert_map_mean:.4f}')
    print(f'  BM25 MAP:         {bm25_map_mean:.4f}')

    # RANKING COMPARISON: Silver vs Gold
    metrics = ['AUC', 'NDCG@10', 'MAP']
    biobert_gold = [biobert_auc, biobert_ndcg_mean, biobert_map_mean]
    bm25_gold = [bm25_auc, bm25_ndcg_mean, bm25_map_mean]

    flips = 0
    total_comparisons = len(metrics)
    for i, m in enumerate(metrics):
        silver_order = 'BioBERT > BM25'
        gold_order = 'BioBERT > BM25' if biobert_gold[i] > bm25_gold[i] else 'BM25 > BioBERT'
        flipped = silver_order != gold_order
        if flipped:
            flips += 1
        status = '*** FLIP ***' if flipped else '(agrees)'
        print(f'  {m}: Silver={silver_order}, Gold={gold_order} {status}')

    empirical_sri = flips / total_comparisons
    print(f'')
    print(f'  === EMPIRICAL SRI = {empirical_sri:.2f} ({flips}/{total_comparisons} metrics flipped) ===')
    if empirical_sri == 0:
        print('  Silver and gold evaluation agree on model ranking across ALL metrics.')
        print('  Model selection under silver labels is VALIDATED by independent gold.')
else:
    print('[SKIP] No TREC data available.')
    biobert_auc = np.nan
    bm25_auc = np.nan
    empirical_sri = np.nan
    biobert_ndcg_mean = np.nan
    bm25_ndcg_mean = np.nan
    biobert_map_mean = np.nan
    bm25_map_mean = np.nan


In [ ]:
# =========================
# 05 - APPENDIX FIGURES
# =========================
if USE_TREC and df_preds is not None and 'biobert_score' in df_preds.columns:
    # FIG E1: Bar chart
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    metrics_names = ['AUC', 'NDCG@10', 'MAP']
    bb_vals = [biobert_auc, biobert_ndcg_mean, biobert_map_mean]
    bm_vals = [bm25_auc, bm25_ndcg_mean, bm25_map_mean]

    for ax, name, bb, bm in zip(axes, metrics_names, bb_vals, bm_vals):
        x = np.arange(2)
        bars = ax.bar(x, [bb, bm], color=['#2ecc71', '#e74c3c'], alpha=0.8, width=0.5)
        ax.set_xticks(x)
        ax.set_xticklabels(['BioBERT\n(silver-trained)', 'BM25\n(baseline)'])
        ax.set_ylabel(name, fontsize=12)
        ax.set_title(f'{name} Under TREC Gold', fontsize=12)
        for bar, val in zip(bars, [bb, bm]):
            ax.text(bar.get_x() + bar.get_width()/2, val + 0.01,
                    f'{val:.3f}', ha='center', fontsize=11, fontweight='bold')
        ax.set_ylim(0, max(bb, bm) * 1.3 + 0.01)

    fig.suptitle(f'Figure E1. Silver-Trained Model vs TREC 2021 Gold Labels (SRI={empirical_sri:.2f})',
                 fontsize=12, y=1.02)
    plt.tight_layout()
    plt.savefig(out_figs / 'figE1_trec_gold_validation.png', dpi=300, bbox_inches='tight')
    plt.close()
    print('[SAVED] figE1_trec_gold_validation.png')

    # FIG E2: Per-patient scatter
    if biobert_ndcg_gold and bm25_ndcg_gold:
        fig, ax = plt.subplots(figsize=(8, 8))
        ax.scatter(bm25_ndcg_gold, biobert_ndcg_gold, alpha=0.6, s=50,
                   c='#3498db', edgecolors='k', linewidths=0.3)
        ax.plot([0, 1], [0, 1], 'k--', alpha=0.3, label='Equal performance')
        ax.set_xlabel('BM25 NDCG@10 (per patient)', fontsize=12)
        ax.set_ylabel('BioBERT NDCG@10 (per patient)', fontsize=12)
        ax.set_title('Figure E2. Per-Patient NDCG@10 Under TREC Gold Labels', fontsize=11)
        ax.legend(fontsize=10)
        ax.grid(True, alpha=0.3)
        bb_wins = sum(1 for b, m in zip(biobert_ndcg_gold, bm25_ndcg_gold) if b > m)
        bm_wins = sum(1 for b, m in zip(biobert_ndcg_gold, bm25_ndcg_gold) if m > b)
        ties = len(biobert_ndcg_gold) - bb_wins - bm_wins
        ax.text(0.05, 0.95, f'BioBERT wins: {bb_wins}\nBM25 wins: {bm_wins}\nTies: {ties}',
                transform=ax.transAxes, fontsize=11, verticalalignment='top',
                bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
        plt.tight_layout()
        plt.savefig(out_figs / 'figE2_per_patient_ndcg_scatter.png', dpi=300, bbox_inches='tight')
        plt.close()
        print('[SAVED] figE2_per_patient_ndcg_scatter.png')
else:
    print('[SKIP] No TREC predictions available for figures.')


In [ ]:
# =========================
# 05 - SAVE RESULTS
# =========================
if USE_TREC and df_preds is not None and 'biobert_score' in df_preds.columns:
    df_results = pd.DataFrame({
        'metric': ['AUC', 'NDCG@10', 'MAP'],
        'biobert_gold': [biobert_auc, biobert_ndcg_mean, biobert_map_mean],
        'bm25_gold': [bm25_auc, bm25_ndcg_mean, bm25_map_mean],
        'silver_ranking': ['BioBERT > BM25'] * 3,
        'gold_ranking': [
            'BioBERT > BM25' if biobert_auc > bm25_auc else 'BM25 > BioBERT',
            'BioBERT > BM25' if biobert_ndcg_mean > bm25_ndcg_mean else 'BM25 > BioBERT',
            'BioBERT > BM25' if biobert_map_mean > bm25_map_mean else 'BM25 > BioBERT',
        ],
        'ranking_agrees': [
            biobert_auc > bm25_auc,
            biobert_ndcg_mean > bm25_ndcg_mean,
            biobert_map_mean > bm25_map_mean,
        ]
    })
    df_results.to_csv(out_tabs / 'results_05_trec_validation.csv', index=False)

    print()
    print('Table E1. TREC 2021 External Validation')
    print('=' * 85)
    print(f'{"Metric":>10s}  {"BioBERT":>10s}  {"BM25":>10s}  {"Silver Rank":>16s}  {"Gold Rank":>16s}  {"Agrees":>7s}')
    print('-' * 85)
    for _, row in df_results.iterrows():
        print(f'{row["metric"]:>10s}  {row["biobert_gold"]:>10.4f}  {row["bm25_gold"]:>10.4f}  '
              f'{row["silver_ranking"]:>16s}  {row["gold_ranking"]:>16s}  {str(row["ranking_agrees"]):>7s}')
    print('=' * 85)
    print(f'Empirical SRI: {empirical_sri:.2f}')
    n_test = len(df_preds)
    n_patients = df_preds["patient_id"].nunique()
    print(f'TREC test pairs: {n_test:,} | TREC patients: {n_patients}')
    print(f'Model trained on: eICU/Jaccard silver labels')
    print(f'Evaluated on: TREC 2021 expert gold labels')

    manifest = write_manifest(out_base, {
        'experiment': EXP,
        'run_id': run_id,
        'use_trec': True,
        'empirical_sri': float(empirical_sri),
        'biobert_auc_gold': float(biobert_auc),
        'bm25_auc_gold': float(bm25_auc),
        'biobert_ndcg10_gold': float(biobert_ndcg_mean),
        'bm25_ndcg10_gold': float(bm25_ndcg_mean),
        'biobert_map_gold': float(biobert_map_mean),
        'bm25_map_gold': float(bm25_map_mean),
        'n_trec_test_pairs': int(n_test),
        'n_trec_patients': int(n_patients),
        'ranking_agreement_all_metrics': bool(empirical_sri == 0),
    })
else:
    manifest = write_manifest(out_base, {
        'experiment': EXP,
        'run_id': run_id,
        'use_trec': False,
    })

print(f'[DONE] Outputs saved under: {out_base}')


In [ ]:
# ==============================
# END OF NOTEBOOK
# ==============================
summary = {
    'run_dir': str(out_base),
    'fig_dir': str(out_figs),
    'tables_dir': str(out_tabs),
}
summary_path = out_base / 'run_summary.json'
summary_path.write_text(json.dumps(summary, indent=2))
print('[DONE] Outputs saved under:', out_base)
